In [ ]:
from jax import random
import json
import pandas as pd
from datetime import datetime, timedelta
from numpyro import distributions as dist
from numpyro import infer
import yaml as yml
import pycountry

from emu_renewal.inputs import BASE_PATH, DATA_PATH, get_standard_priors, get_country_vacc_data, get_country_vars, find_relevant_vars
from emu_renewal.inputs import get_worldbank_national_pop, get_country_mobility, get_standard_targets, get_euro_var_inputs, get_row_proportions
from emu_renewal.process import CosineMultiCurve
from emu_renewal.distributions import GammaDens
from emu_renewal.renew import MultiStrainModel
from emu_renewal.outputs import store_outputs
from emu_renewal.calibration import StandardCalib
from emu_renewal.targets import StandardDispTarget

In [ ]:
proc_update_freq = 7
init_duration = 50
seed_duration = 10
analysis_to_data_delay = 14
iterations = 1000
priors = get_standard_priors()
initial_countries = json.load(open(DATA_PATH / f"config/countries.json", "r"))
all_countries = initial_countries["admissions"] + initial_countries["occupancy"]

In [ ]:
for country in ["Italy"]:
    print(f"Running {country}")
    end_time = datetime(2021, 12, 31)
    analysis_time = datetime.now().strftime("%Y%m%d_%H%M")

    iso2 = pycountry.countries.lookup(country).alpha_2
    iso3 = pycountry.countries.lookup(country).alpha_3
    country_name = pycountry.countries.lookup(country).name
    pop = get_worldbank_national_pop(iso3)
    hosp_indicator, hosp_colour = ("Weekly new hospital admissions", "black") if iso2 in initial_countries["admissions"] else ("Daily hospital occupancy", "red")
    cases_target, hosp_target, deaths_target, seroprev_target, init_data = get_standard_targets(iso2, datetime(2020, 1, 1), end_time, 50, hosp_indicator)
    vacc_data = get_country_vacc_data(country)
    per_capita_deaths = deaths_target / pop
    death_start_threshold = 2e-6
    start_time = per_capita_deaths.index[per_capita_deaths.gt(death_start_threshold)].min()
    mobility = get_country_mobility(iso3)
    var_country_name = pycountry.countries.lookup(iso3).official_name if iso3 in ["CZE"] else pycountry.countries.lookup(iso3).name
    var_data = get_country_vars(var_country_name)
    relevant_vars = find_relevant_vars(var_data, end_time - timedelta(270), 10)
    select_var_data = var_data[relevant_vars]
    var_props = get_row_proportions(select_var_data)
    pre_alpha_vars = ["20A.EU1"] if iso3 == "LTU" else ["20A.EU1", "20A.EU2"]
    eu_prop = select_var_data[pre_alpha_vars].sum(axis=1) / select_var_data.sum(axis=1)
    vacc_data = get_country_vacc_data(country)
    cov_threshold = 0.05
    end_time = vacc_data[vacc_data.gt(cov_threshold * 100)].idxmin()

    # Collate targets
    seroprev_target_dict = {"seropos": StandardDispTarget(seroprev_target, weight=20.0)} if any(seroprev_target) else {}
    all_targets = {
        "weekly_cases": StandardDispTarget(cases_target, weight=20.0),
        "weekly_deaths": StandardDispTarget(deaths_target, weight=20.0),
        "prop_eu": StandardDispTarget(eu_prop, weight=20.0),
        "occupancy": StandardDispTarget(hosp_target, weight=20.0),
    } | seroprev_target_dict
    
    # for mob_analysis_type in mobility.columns:
    #     print(mob_analysis_type)
        
    #     # Define model and fitter
    #     model = MultiStrainModel(
    #         pop,
    #         start_time,
    #         end_time,
    #         proc_update_freq,
    #         CosineMultiCurve(),
    #         GammaDens(),
    #         init_duration,
    #         init_data,
    #         GammaDens(),
    #         GammaDens(),
    #         strains,
    #         "eu",
    #         seed_times,
    #         100.0,
    #         mobility[mob_analysis_type].dropna(),
    #     )
        
    #     # Run calibration
    #     calib = StandardCalib(model, priors, all_targets, proc_dispersion=dist.HalfNormal(0.5))
    #     kernel = infer.NUTS(calib.calibration, dense_mass=True, init_strategy=calib.custom_init(radius=0.1))
    #     mcmc = infer.MCMC(kernel, num_chains=4, num_samples=iterations, num_warmup=iterations)
    #     mcmc.run(random.PRNGKey(0), extra_fields=["potential_energy"])
    #     store_outputs(country, mob_analysis_type, analysis_time, model, calib, mcmc)